# 免费历史预期覆盖扩展

固定2018年至2026-09-25三家公司105次财报日历，不按收益挑事件。NVDA日期与已冻结的原始公司披露核对；AMD/INTC日期仍需交叉确认。只做来源资格和实际值匹配，不训练或声称超额。

In [1]:
from pathlib import Path
import json,hashlib,os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import qlib,pandas as pd,yaml
from qlib.data import D
from qlib.workflow import R
from src.data.refinitiv_newsletter import collect_newsletters
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/refinitiv_newsletter_extended_20261013';plan=json.loads((card/'plan.json').read_text())
for path,digest in plan['calendar_source_hashes'].items():assert hashlib.sha256((root/path).read_bytes()).hexdigest()==digest
qlib.init(provider_uri=str(root/plan['price_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(freq='day')).normalize().unique().sort_values()
assert all(pd.Timestamp(e['report_date']) in calendar for e in plan['events'])
assert hashlib.sha256((root/plan['nvda_fact_manifest']).read_bytes()).hexdigest()==plan['nvda_fact_manifest_sha256']
raw=root/plan['raw_cache'];manifest=collect_newsletters(raw,plan);events=pd.DataFrame(manifest['entries'])
audit={'expected_events':len(plan['events']),'parsed_events':len(events),'failures':manifest['failures'],'per_symbol':events.groupby('symbol').size().to_dict() if len(events) else {},'prior_day_document_clock_rows':int(events.document_clock_before_report_day.sum()) if len(events) else 0,'transport_certificate_verified_rows':int(events.tls_certificate_verified.sum()) if len(events) else 0,'publication_clock_certified':False,'point_in_time_consensus_certified':False,'model_fits':0,'backtests':0}
issuer_raw=(raw/'nvda_20220808_preliminary.html').read_bytes()
issuer_warning={'url':'https://nvidianews.nvidia.com/news/nvidia-announces-preliminary-financial-resultsfor-second-quarter-fiscal-2023','sha256':hashlib.sha256(issuer_raw).hexdigest(),'published_date':'2022-08-08','preliminary_revenue_million_usd':6700,'prior_outlook_million_usd':8100,'later_consensus_million_usd':float(events.loc[(events.symbol=='NVDA') & (events.report_date=='2022-08-24'),'consensus_revenue_million_usd'].iloc[0]),'interpretation':'later newsletter retains consensus above already-published warning; do not label repeated bad news as novel on actual earnings date'}
audit['revenue_estimate_available_rows']=int(events.consensus_revenue_million_usd.notna().sum())
audit['smart_estimate_available_rows']=int(events.smart_eps.notna().sum())
audit['preannouncement_control_required']=True
audit['calendar_corrections']=plan['calendar_corrections']
audit['amd_intc_issuer_dates_corroborated']=False
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],recorder_name='public_ibes_newsletter_extended_coverage',uri=str(tracking)):
    rec=R.get_recorder();rec.log_params(plan_sha256=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),model_fits=0,backtests=0)
    rec.save_objects(artifact_path='sources',manifest=manifest,pdfs={p.name:p.read_bytes() for p in raw.glob('*.pdf')},parser=(root/'src/data/refinitiv_newsletter.py').read_text(),issuer_preliminary_html=issuer_raw,issuer_warning_audit=issuer_warning)
    rec.save_objects(artifact_path='analysis',events=events,audit=audit,calendar=calendar)
    rec.log_metrics(source_events=len(events),registered_artifact_count=8)
summary={'experiment':plan['experiment'],'decision':'source_coverage_complete_needs_event_and_publication_qualification' if len(events)==len(plan['events']) else 'needs_source_repair','source_audit':audit,'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'provider_uri':str(root/plan['price_provider']),'region':'us','as_of':'2026-10-09','qlib_docs':'https://qlib.readthedocs.io/en/latest/component/workflow.html','objective_achieved':False,'goal_remains_active':True,'promotion_eligible':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['sources/*','analysis/*']}}
artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';verification={'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]}
(card/'verification_manifest.json').write_text(json.dumps(verification,indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id))
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':plan['experiment'],'provider_uri':summary['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']}));events.to_csv(card/'source_inventory.csv',index=False)
audit


{'expected_events': 105,
 'parsed_events': 105,
 'failures': [],
 'per_symbol': {'AMD': 35, 'INTC': 35, 'NVDA': 35},
 'prior_day_document_clock_rows': 99,
 'transport_certificate_verified_rows': 0,
 'publication_clock_certified': False,
 'point_in_time_consensus_certified': False,
 'model_fits': 0,
 'backtests': 0,
 'revenue_estimate_available_rows': 99,
 'smart_estimate_available_rows': 99,
 'preannouncement_control_required': True,
 'calendar_corrections': [],
 'amd_intc_issuer_dates_corroborated': False}

# 免费历史预期来源覆盖扩展

固定2018年至2026-09-25的NVDA、AMD、INTC共105次财报日历，已恢复105条EPS普通一致预期，各公司35条；99条含营收预期和StarMine SmartEstimate。原始PDF及失败修复记录均保存NAS Qlib Recorder，0模型、0回测、没有新增收益结论。

NVDA的35个日期已和冻结原始公司披露事实核对，与供应商日历一致；计划同时保留原始公司报告的季度营收及FY季度标识，供下一步逐期口径核对。AMD/INTC仍是供应商报告日期，未标记公司原文认证。Qlib原生真实交易日历用于日期校验，没有用单标的历史合成另一标的价格。

修复了旧Thomson Reuters与新LSEG的I/B/E/S脚注差异、公司全称、跨页表格和负EPS的“Expected loss”表达。早期阶段58/105、101/105的Recorder及原始缺失均保留，未修改来源金额。6次使用同供应商晨报补充：AMD2021-10-26、2022-08-02、2024-10-29、2025-08-05、2026-08-04，以及INTC2023-04-27。晨报只有EPS，其营收和SmartEstimate均保持未知，不从别的版本补数。

99份使用报告日前生成的DayAhead，6份使用报告日晨报；文件名不等于发布时间，文档元数据不等于公开可用证明。所有publication_clock_certified、point_in_time_consensus_certified、promotion_eligible为false。公开旧域名证书过期的单次读取限制继续记录，未改全局TLS设置。

已知NVDA2022-08的提前预警案例说明：即使预期快照本身带日期，实际值减预期仍可能混入早已公开的消息。下一步必须把原始实际值、已发生的预警/修订及季度口径接入同一时间线；未经该检查，不把105条快照称为105个新信息冲击，不重跑旧EPS方案冒充新研究。

决定：source_coverage_complete_needs_event_and_publication_qualification。本轮完成了既定时间段覆盖，剩余资格为发布时间、公司日期及实际值/预警口径。稳定净SOXX超额目标未达成，生产SIM未改。下一步先完成实际值和信息新鲜度合同，再冻结有限信息增量检验。
